In [1]:
import json
import re
import html
import pandas as pd
from html.parser import HTMLParser
import os

In [18]:
class HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, d):
        self.parts.append(d)

    def get_data(self):
        return "".join(self.parts)


def strip_html(value):
    if value is None or pd.isna(value):
        return ""
    text = html.unescape(str(value))
    parser = HTMLStripper()
    parser.feed(text)
    text = parser.get_data().replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return html.unescape(str(value)).strip()


def smart_join(parts):
    parts = [p.strip().rstrip(".") for p in parts if p and str(p).strip()]
    return ". ".join(parts)


def format_mdy(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return ""
    s = str(value).split("T")[0].strip()
    dt = pd.to_datetime(s, errors="coerce")
    if pd.isna(dt):
        return s
    return f"{dt.month}/{dt.day}/{dt.year}"


def build_whitney_medium(attributes):
    parts = [
        clean_text(attributes.get("medium")),
        clean_text(attributes.get("classification")),
    ]
    return " | ".join([p for p in parts if p])


def build_whitney_alt_text(attributes):
    return smart_join([
        strip_html(attributes.get("alt_text")),
        strip_html(attributes.get("ai_alt_text")),
    ])


def build_whitney_description(attributes):
    visual_description = strip_html(attributes.get("visual_description"))
    object_label = strip_html(attributes.get("object_label"))
    description = strip_html(attributes.get("description"))

    if visual_description:
        return visual_description
    if object_label:
        return object_label
    return description


def get_first_image_url(attributes):
    images = attributes.get("images") or []
    if images and isinstance(images, list):
        return clean_text(images[0].get("url"))
    return ""

In [19]:
def convert_whitney_json_to_artworks(json_path, artist_id, institution_id=1, output_csv=None):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []

    for item in data.get("data", []):
        attributes = item.get("attributes", {})

        records.append({
            "id": "",
            "artist": artist_id,
            "title": clean_text(attributes.get("title")),
            "alt_text": build_whitney_alt_text(attributes),
            "description": build_whitney_description(attributes),
            "date_created": clean_text(attributes.get("display_date")),
            "date_acquired_or_updated": format_mdy(attributes.get("created_at")),
            "institution": institution_id,
            "medium": build_whitney_medium(attributes),
            "image_url": get_first_image_url(attributes),
        })

    converted = pd.DataFrame(records, columns=[
        "id",
        "artist",
        "title",
        "alt_text",
        "description",
        "date_created",
        "date_acquired_or_updated",
        "institution",
        "medium",
        "image_url",
    ])

    if output_csv:
        if os.path.exists(output_csv):
            converted.to_csv(output_csv, mode="a", header=False, index=False)
        else:
            converted.to_csv(output_csv, index=False)

    return converted

In [20]:
artist_name = "Roger Shimomura"

ARTIST_ID = 25
INSTITUTION_ID = 1
INPUT_JSON = f"./exports/whitney.json"
OUTPUT_CSV = f"./formatted-exports/{artist_name}.csv"

converted = convert_whitney_json_to_artworks(
    json_path=INPUT_JSON,
    artist_id=ARTIST_ID,
    institution_id=INSTITUTION_ID,
    output_csv=OUTPUT_CSV
)

print(converted.to_csv(index=False))

id,artist,title,alt_text,description,date_created,date_acquired_or_updated,institution,medium,image_url
,25,Yellow No Same,,"Roger Shimomura, Yellow No Same, 1992. See individual records. Series of twelve color screenprints., see individual records. Whitney Museum of American Art, New York; purchase with funds from the Print Committee 2003.260.1-12",1992,11/4/2019,1,See individual records. Series of twelve color screenprints. | Prints,
,25,American Guardian,A silhouetted soldier with rifle watches over a guarded camp of barracks and watchtower,"Roger Shimomura, American Guardian, 2007. Lithograph, sheet: 31 5/8 × 43 1/8 in. (80.3 × 109.5 cm) Image: 27 1/8 × 39 in. (68.9 × 99.1 cm). Whitney Museum of American Art, New York; purchase, with funds from the Print Committee 2008.41. ©Roger Shimomura",2007,8/30/2017,1,Lithograph | Prints,https://whitneymedia.org/assets/artwork/33728/2008_41_cropped.jpg
,25,Yellow No Same,Two children and a man peer through barbed wire with serious expressions